# Lab 01 — Boto3 Basics: Ingesting Real Data into S3

In this notebook you'll learn how to use **boto3** (the Python SDK for S3) by connecting to
a local RustFS instance. Instead of creating dummy files, we'll work with a **real dataset**
downloaded from the internet and ingest it into the **bronze** layer of our Data Lake.

> **What is the Bronze layer?** In the [Medallion Architecture](https://www.databricks.com/glossary/medallion-architecture),
> data flows through three quality tiers:
> - **Bronze** — raw data, exactly as received from the source (no changes)
> - **Silver** — cleaned, validated, and deduplicated data
> - **Gold** — aggregated, business-ready data

### What you'll practice

| # | Operation | boto3 API |
|---|-----------|----------|
| 1 | Connect to S3 | `boto3.client()` |
| 2 | List buckets | `list_buckets()` |
| 3 | **Scenario A** — Download from web → local staging → upload to S3 bronze | `upload_file()` |
| 4 | **Scenario B** — Download from web and stream directly to S3 (no disk I/O) | `put_object()` |
| 5 | List and inspect objects in a bucket | `list_objects_v2()`, `head_object()` |
| 6 | Download an object from S3 | `get_object()` |
| 7 | Generate a pre-signed URL | `generate_presigned_url()` |
| 8 | Cleanup | `delete_object()` |

---
## 0 · Prerequisites

Before running this notebook, make sure:

1. RustFS is running: `docker compose up -d`
2. The `bronze`, `silver`, and `gold` buckets were created by the `rustfs-init` service
3. The virtual environment is active with `boto3` installed (`uv sync`)

---
## 1 · Connect to RustFS (S3-compatible)

`boto3.client('s3')` creates an **S3 client** that communicates over HTTP with any
S3-compatible storage. We need to provide:

- **endpoint_url** — the RustFS address (port 9000)
- **access_key / secret_key** — credentials configured in `docker-compose.yml`
- **region_name** — required by the S3 protocol, but irrelevant for local storage
- **config** — two critical settings for non-AWS endpoints:
  - `signature_version='s3v4'` — forces **Signature Version 4** (HMAC-SHA256). Non-AWS S3-compatible
    stores like RustFS, MinIO, and Ceph reject the older V2 signatures.
  - `addressing_style='path'` — uses path-style URLs (`/bucket/key`) instead of
    virtual-hosted-style (`bucket.host/key`), which doesn't work with `localhost`.

> **Production note:** Never hardcode credentials in source code. Use environment variables,
> AWS profiles (`~/.aws/credentials`), or a secrets manager like AWS Secrets Manager or HashiCorp Vault.

In [ ]:
import boto3
from botocore.config import Config

s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
    config=Config(
        signature_version='s3v4',           # required for RustFS / MinIO / non-AWS endpoints
        s3={'addressing_style': 'path'},     # path-style: localhost:9000/bucket/key
    ),
)

print('✅ Connected to RustFS S3!')

---
## 2 · List existing buckets

`list_buckets()` returns a dictionary with a `'Buckets'` key — a list of all available
buckets. The `bronze`, `silver`, and `gold` buckets were created automatically by
`docker-compose.yml`.

In [ ]:
response = s3.list_buckets()

print('📦 Available buckets:')
for b in response['Buckets']:
    print(f"   • {b['Name']}  (created at {b['CreationDate']})")

---
## 📊 Dataset

We'll use the **Iris Dataset** — one of the most classic datasets in data science.
It contains 150 flower samples with 4 measurements (sepal and petal dimensions)
and the species label.

Public URL we'll use:

```
https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv
```

This CSV is only **~4 KB** — perfect for a hands-on lab.

In [ ]:
# Public URL for the Iris dataset (hosted on seaborn's GitHub)
DATASET_URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv'

# Destination bucket — the bronze layer of our Data Lake
BRONZE_BUCKET = 'bronze'

---
## 3 · Scenario A — Web → Local Staging → S3 Bronze

This is the most common pattern in data pipelines:

```
Internet  ──(download)──▶  temp/staging/  ──(upload)──▶  s3://bronze/
```

### Why use local staging?

- **Validation**: inspect or validate the file before sending it to S3
- **Retry**: if the upload fails, the file is already on disk — just try again
- **Transformation**: in real pipelines, you often need to decompress, rename,
  or convert the file before uploading

### Step 3.1 — Download the CSV from the web to local staging

We use the standard library `urllib` to avoid extra dependencies.
The file will be saved at `temp/staging/iris.csv`.

In [ ]:
import urllib.request
from pathlib import Path

# Local staging directory (already exists in the project)
staging_dir = Path('../temp/staging')
staging_dir.mkdir(parents=True, exist_ok=True)

local_file = staging_dir / 'iris.csv'

# Download the CSV from the internet
urllib.request.urlretrieve(DATASET_URL, local_file)

# Confirm the file was saved
size_bytes = local_file.stat().st_size
print(f'⬇️  File downloaded to: {local_file}')
print(f'   Size: {size_bytes:,} bytes')
print()

# Peek at the first 5 lines
print('👀 First 5 lines of the CSV:')
for line in local_file.read_text().splitlines()[:5]:
    print(f'   {line}')

### Step 3.2 — Upload from local staging to the bronze bucket

We use `s3.upload_file()`, the simplest way to send a **file from disk** to S3.
Parameters:

| Parameter | Meaning |
|-----------|--------|
| `Filename` | Local path of the file |
| `Bucket` | Target bucket name |
| `Key` | Path/name of the object inside the bucket |

> **Note:** The **Key** can contain `/` to simulate a directory structure — e.g., `raw/iris/iris.csv`.
> S3 has no real directories; the `/` is just part of the key name, but most tools display it as a folder.

In [ ]:
# S3 Key — organized by source and dataset name
s3_key_a = 'iris_via_staging.csv'

s3.upload_file(
    Filename=str(local_file),   # local file path
    Bucket=BRONZE_BUCKET,       # target bucket
    Key=s3_key_a,               # object path inside the bucket
)

print('✅ Upload complete!')
print(f'   Source:      {local_file}')
print(f'   Destination: s3://{BRONZE_BUCKET}/{s3_key_a}')

---
## 4 · Scenario B — Web → S3 Bronze (direct streaming, no disk)

This pattern is called **diskless streaming** — bytes flow directly from the internet
to S3, never touching the local disk:

```
Internet  ──(bytes in memory)──▶  s3://bronze/
```

### When to prefer this approach

- **Disk savings**: no local space needed (essential in lightweight containers)
- **Speed**: eliminates the write/read cycle on disk
- **Simplicity**: fewer moving parts, fewer failure points

### How it works

1. `urllib.request.urlopen()` reads the file bytes into memory
2. Those bytes are passed directly to `s3.put_object(Body=...)` — no temp file needed

In [ ]:
import urllib.request

# 1. Download bytes from the internet into memory
with urllib.request.urlopen(DATASET_URL) as response:
    csv_bytes = response.read()

print(f'⬇️  Bytes read from internet: {len(csv_bytes):,}')

# 2. Send directly to S3 (no disk write)
s3_key_b = 'iris_via_stream.csv'

s3.put_object(
    Bucket=BRONZE_BUCKET,
    Key=s3_key_b,
    Body=csv_bytes,              # raw bytes → S3
    ContentType='text/csv',      # optional but good practice: helps clients know the format
)

print('✅ Direct upload complete!')
print(f'   Destination: s3://{BRONZE_BUCKET}/{s3_key_b}')

---
## 5 · Inspect objects in the bronze bucket

Let's list objects in the `bronze` bucket using `list_objects_v2()`. We filter by the
prefix `` to see only our uploads.

Then we use `head_object()` to inspect an object's **metadata** without downloading
its content — useful for checking size, MIME type, last-modified date, and integrity
hash (ETag), all in a single lightweight request.

In [ ]:
# 5.1 — List objects with the raw/iris/ prefix
response = s3.list_objects_v2(Bucket=BRONZE_BUCKET, Prefix='')

print('📂 Objects in s3://bronze')
print('-' * 60)
for obj in response.get('Contents', []):
    print(f"   {obj['Key']:40s}  {obj['Size']:>6,} bytes")

print()

# 5.2 — Inspect object metadata without downloading it (head_object)
meta = s3.head_object(Bucket=BRONZE_BUCKET, Key=s3_key_b)

print(f'🔍 Metadata for {s3_key_b}:')
print(f'   Content-Type:   {meta["ContentType"]}')
print(f'   Content-Length: {meta["ContentLength"]:,} bytes')
print(f'   Last-Modified:  {meta["LastModified"]}')
print(f'   ETag:           {meta["ETag"]}')

---
## 6 · Download an object from S3 and verify its content

`get_object()` returns a dictionary with a `'Body'` key — a **streaming response**.
We call `.read()` to get the raw bytes and `.decode()` to convert them to a string.

Let's download the file uploaded in Scenario B and verify it arrived intact.

In [ ]:
response = s3.get_object(Bucket=BRONZE_BUCKET, Key=s3_key_b)
body = response['Body'].read().decode('utf-8')

# Display the first 6 lines to verify the content
lines = body.splitlines()
print(f'📄 Content of s3://{BRONZE_BUCKET}/{s3_key_b}')
print(f'   Total lines: {len(lines)} (1 header + {len(lines)-1} records)\n')

for line in lines[:6]:
    print(f'   {line}')

---
## 7 · Generate a Pre-signed URL

A **pre-signed URL** lets anyone download an S3 object **without needing credentials**.
The URL embeds a cryptographic signature and an expiration time.

Common use cases:
- Temporarily sharing a report with a colleague
- Letting a frontend download a file without exposing your S3 credentials to the client
- Triggering a one-time file download from a third-party service

> **Signature versions:** boto3 can sign URLs with **V2** (legacy, HMAC-SHA1) or **V4** (current, HMAC-SHA256).
> Non-AWS stores like RustFS and MinIO only accept V4. That's why `signature_version='s3v4'` in the client
> config (Step 1) is essential — it ensures both regular API calls *and* pre-signed URLs use V4.

> ⚠️ Since we're using local RustFS, this URL only works on your machine.

In [ ]:
url = s3.generate_presigned_url(
    ClientMethod='get_object',
    Params={'Bucket': BRONZE_BUCKET, 'Key': s3_key_b},
    ExpiresIn=3600,  # valid for 1 hour
)

print('🔗 Pre-signed URL (valid for 1h):')
print(f'   {url}')
print()
print('Try it in your terminal:')
print('   curl -s "<url above>" | head -5')

---
## 8 · Cleanup

Best practice: delete test objects at the end of a lab to keep the bucket clean.

We'll also remove the local staging file.

In [ ]:
# Delete both objects we created
for key in [s3_key_a, s3_key_b]:
    s3.delete_object(Bucket=BRONZE_BUCKET, Key=key)
    print(f'🗑️  Deleted: s3://{BRONZE_BUCKET}/{key}')

# Remove the local staging file
if local_file.exists():
    local_file.unlink()
    print(f'🗑️  Removed: {local_file}')

print('\n✅ Cleanup complete!')

---
## 📋 Summary

| Scenario | Flow | Main API | When to use |
|----------|------|----------|------------|
| **A** — Local staging | Web → disk → S3 | `upload_file()` | When you need to validate, transform, or retry uploads |
| **B** — Direct streaming | Web → memory → S3 | `put_object()` | When you want speed and disk savings |

### API cheat sheet

| Method | Purpose |
|--------|---------|
| `list_buckets()` | List all available buckets |
| `list_objects_v2()` | List objects in a bucket (supports prefix filtering) |
| `upload_file()` | Upload a local file to S3 |
| `put_object()` | Upload raw bytes or a file-like object to S3 |
| `head_object()` | Fetch object metadata without downloading content |
| `get_object()` | Download object content as a stream |
| `generate_presigned_url()` | Create a temporary, credential-free download link |
| `delete_object()` | Delete a single object from S3 |

### Next steps

- **Lab 02**: Multipart upload for large files (> 100 MB)
- **Lab 03**: Object versioning and retention policies